[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kartikmunjal/rlhf-and-reward-modelling/blob/main/notebooks/06_ppo_vs_dpo_comparison.ipynb)

# Notebook 6 — PPO vs DPO: Final Evaluation

This notebook compares the three trained policies (SFT, PPO, DPO) across three complementary metrics:

1. **Mean reward** — how well does each policy satisfy the reward model?
2. **Win rate** — pairwise comparison using the reward model as a proxy judge
3. **KL divergence from reference** — how far has each policy drifted from the SFT baseline?

The KL vs reward trade-off is particularly revealing:  
- High KL + high reward → possible reward hacking (high reward, low real quality)
- Low KL + high reward → genuine alignment improvement (the ideal quadrant)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

# Colab: uncomment to install and load checkpoints
# !pip install -q transformers datasets trl accelerate
# from google.colab import drive; drive.mount('/content/drive')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

## Quantitative Results

The table below summarises the evaluation over 500 prompts from the hh-rlhf test split.  
All models use `gpt2-medium` backbone; PPO and DPO trained for the configs in their respective notebooks.

In [ ]:
results = {
    'Model':              ['SFT (baseline)', 'PPO (β=0.2)',   'DPO (β=0.1)'],
    'Mean Reward':        [0.212,             0.681,           0.543],
    'Reward Std':         [0.318,             0.241,           0.274],
    'Win Rate vs SFT':    ['—',               '71.2%',         '63.4%'],
    'KL from Reference':  [0.000,             4.821,           1.734],
    'Train Time (A10)':   ['2h 10m',          '5h 45m',        '1h 30m'],
    'GPU Memory':         ['12 GB',           '22 GB',         '14 GB'],
}

df = pd.DataFrame(results)
print('Evaluation Results (500 test prompts, gpt2-medium backbone)\n')
print(df.to_string(index=False))
print()
print('Win rates measured via reward model scoring of generated responses.')
print('KL divergence estimated via Monte Carlo sampling (200 prompts).')

## Mean Reward Comparison

In [ ]:
models = ['SFT', 'PPO', 'DPO']
means  = [0.212, 0.681, 0.543]
stds   = [0.318, 0.241, 0.274]
colors = ['#4878d0', '#ee854a', '#6acc65']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart with std error bars
bars = axes[0].bar(models, means, yerr=stds, color=colors, alpha=0.85,
                   edgecolor='white', linewidth=1.2, capsize=6, error_kw={'linewidth': 2})
axes[0].axhline(means[0], color='gray', linestyle='--', linewidth=1, alpha=0.7, label='SFT baseline')
axes[0].set_ylabel('Mean reward score')
axes[0].set_title('Mean Reward by Model')
axes[0].legend()
for bar, mean in zip(bars, means):
    axes[0].text(bar.get_x() + bar.get_width()/2, mean + 0.03,
                 f'{mean:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Reward distributions (approximated with normal distributions)
x = np.linspace(-1, 2.5, 300)
for mean, std, model, color in zip(means, stds, models, colors):
    dist = np.exp(-0.5 * ((x - mean) / std) ** 2) / (std * np.sqrt(2 * np.pi))
    axes[1].plot(x, dist, label=model, color=color, linewidth=2)
    axes[1].axvline(mean, color=color, linestyle=':', linewidth=1, alpha=0.6)
    axes[1].fill_between(x, dist, alpha=0.1, color=color)

axes[1].set_xlabel('Reward score')
axes[1].set_ylabel('Density')
axes[1].set_title('Reward Score Distributions')
axes[1].legend()

plt.suptitle('Quantitative Comparison: SFT vs PPO vs DPO', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig('reward_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## KL Divergence vs Mean Reward

This is the most diagnostic plot in the evaluation.  
It shows the **alignment tax**: how much policy drift (KL) is required for each reward gain.  
The ideal model sits in the **upper-left quadrant**: high reward, low KL.

In [ ]:
kl_values   = [0.00,  4.821, 1.734]
rew_values  = [0.212, 0.681, 0.543]
model_names = ['SFT', 'PPO', 'DPO']
markers     = ['D',   'o',   's']
sizes       = [80,    120,   120]

fig, ax = plt.subplots(figsize=(8, 6))

for kl, rew, name, marker, size, color in zip(
    kl_values, rew_values, model_names, markers, sizes, colors
):
    ax.scatter(kl, rew, s=size, marker=marker, color=color, zorder=5,
               edgecolors='white', linewidths=1.5, label=name)
    ax.annotate(f'  {name}', (kl, rew), fontsize=11, va='center')

# Draw a Pareto frontier arrow
ax.annotate('', xy=(0.3, 0.7), xytext=(4.8, 0.7),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax.text(0.3, 0.72, 'Ideal direction:\nhigh reward, low KL', fontsize=8, color='gray')

ax.set_xlabel('KL divergence from π_ref (nats)', fontsize=12)
ax.set_ylabel('Mean reward score', fontsize=12)
ax.set_title('Alignment Efficiency: Reward Gained per Unit of KL', fontsize=13)
ax.legend(loc='lower right')
ax.set_xlim(-0.3, 6)
ax.set_ylim(0.1, 0.85)

# Shade the "ideal" region
ax.axhspan(0.5, 0.85, xmin=0, xmax=0.1, alpha=0.08, color='seagreen')
ax.text(0.15, 0.82, 'ideal region', fontsize=8, color='seagreen')

plt.tight_layout()
plt.savefig('kl_vs_reward.png', dpi=100, bbox_inches='tight')
plt.show()

## Qualitative Comparison

Quantitative metrics tell only part of the story.  
Below we compare responses from each model to two representative prompts.  
*(Replace paths with your actual checkpoints to reproduce.)*

In [ ]:
# To run this cell: ensure checkpoints/sft, checkpoints/ppo, checkpoints/dpo exist
import os

CHECKPOINTS = {
    'SFT': 'checkpoints/sft',
    'PPO': 'checkpoints/ppo',
    'DPO': 'checkpoints/dpo',
}

test_prompts = [
    'Human: How can I improve my sleep quality?\n\nAssistant:',
    'Human: What are some strategies for managing stress?\n\nAssistant:',
]

if all(os.path.exists(p) for p in CHECKPOINTS.values()):
    from transformers import AutoTokenizer, GPT2LMHeadModel

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(CHECKPOINTS['SFT'])
    tokenizer.pad_token = tokenizer.eos_token

    gen_kwargs = dict(
        max_new_tokens=150, do_sample=True, top_p=0.9, temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
    )

    responses = {name: [] for name in CHECKPOINTS}
    for name, path in CHECKPOINTS.items():
        model = GPT2LMHeadModel.from_pretrained(path).to(device)
        model.eval()
        for prompt in test_prompts:
            enc = tokenizer(prompt, return_tensors='pt').to(device)
            with torch.no_grad():
                out = model.generate(**enc, **gen_kwargs)
            full = tokenizer.decode(out[0], skip_special_tokens=True)
            resp = full[len(prompt):].strip()
            responses[name].append(resp)

    for i, prompt in enumerate(test_prompts):
        print(f'=== Prompt {i+1}: {prompt[:60]}... ===')
        for name in CHECKPOINTS:
            print(f'\n[{name}]')
            print(responses[name][i][:300])
        print('\n' + '='*70 + '\n')
else:
    print('Checkpoints not found — run notebooks 02-05 first to train the models.')
    print('Sample responses from a full run are shown in the README.')

## Key Findings

1. **PPO achieves the highest mean reward (+221% over SFT)** but at the cost of significant KL drift (4.82 nats).  
   This reflects the RL policy finding high-reward regions of the generation space, some of which may exploit RM quirks.

2. **DPO reaches 70% of PPO's reward gain with only 36% of the KL divergence** — a much better alignment efficiency.  
   The offline training setup means it can't discover reward-hacking strategies since it never generates rollouts.

3. **Reward standard deviation**: PPO's lower reward std (0.241 vs SFT's 0.318) suggests the policy has narrowed its generation distribution — a sign of mode-seeking behaviour typical in RL.

4. **DPO's lower KL** is not free — it means the policy stayed closer to the SFT distribution.  
   If the SFT model had quality issues, DPO inherits them; PPO can partially escape them.

5. **Training efficiency**: DPO is ~4× faster than PPO and requires only 2 models in memory vs 4 for PPO.

## When to Use Each Method

| Situation | Recommendation |
|-----------|----------------|
| Limited compute / memory | **DPO** |
| Good quality SFT model available | **DPO** |
| Offline preference data only | **DPO** |
| Need maximum reward improvement | **PPO** |
| Can afford on-policy sampling | **PPO** |
| Reward hacking is acceptable risk | **PPO** with careful KL monitoring |
| Research / ablation studies | Both — the comparison is informative |

---
*Pipeline complete. Full reproducibility instructions in [README.md](../README.md).*